## Define the Dataset

The dataset is defined as a list of documents and their corresponding class labels (SPAM or HAM).

# Naïve Bayes Classification: Manual and Scikit-Learn Implementation

This notebook demonstrates manual and scikit-learn-based Naïve Bayes classification on a small SPAM/HAM dataset, as described in the exercise. Manual is 1-5 while Scikit-Learn is 6-7

In [1]:
# Define the dataset as a list of (document, class) tuples

documents = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

import pandas as pd

df = pd.DataFrame(documents, columns=["doc", "class"])
df

,doc,class
0,Free money now!!!,SPAM
1,"Hi mom, how are you?",HAM
2,Lowest price for your meds,SPAM
3,Are we still on for dinner?,HAM
4,Win a free iPhone today,SPAM
5,Let's catch up tomorrow at the office,HAM
6,Meeting at 3 PM tomorrow,HAM
7,"Get 50% off, limited time!",SPAM
8,Team meeting in the office,HAM
9,Click here for prizes!,SPAM


## 1. Generate Bag of Words (Word Frequency)

Documents will be tokenized and count word frequencies for each class (HAM and SPAM), building a vocabulary and frequency tables.

In [2]:
import re
from collections import defaultdict, Counter

def tokenize(text):
    # Lowercase and split on non-word characters
    return re.findall(r'\b\w+\b', text.lower())

# Separate documents by class
docs_by_class = {'HAM': [], 'SPAM': []}
for doc, label in documents:
    docs_by_class[label].append(doc)

# Build vocabulary and word counts for each class
vocab = set()
word_counts = {'HAM': Counter(), 'SPAM': Counter()}

for label in docs_by_class:
    for doc in docs_by_class[label]:
        tokens = tokenize(doc)
        vocab.update(tokens)
        word_counts[label].update(tokens)

print("Vocabulary:", sorted(vocab))
print("\nHAM word counts:", word_counts['HAM'])
print("\nSPAM word counts:", word_counts['SPAM'])

Vocabulary: ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

HAM word counts: Counter({'the': 3, 'are': 2, 'you': 2, 'tomorrow': 2, 'at': 2, 'office': 2, 'meeting': 2, 'hi': 1, 'mom': 1, 'how': 1, 'we': 1, 'still': 1, 'on': 1, 'for': 1, 'dinner': 1, 'let': 1, 's': 1, 'catch': 1, 'up': 1, '3': 1, 'pm': 1, 'team': 1, 'in': 1, 'can': 1, 'send': 1, 'report': 1})

SPAM word counts: Counter({'free': 2, 'for': 2, 'money': 1, 'now': 1, 'lowest': 1, 'price': 1, 'your': 1, 'meds': 1, 'win': 1, 'a': 1, 'iphone': 1, 'today': 1, 'get': 1, '50': 1, 'off': 1, 'limited': 1, 'time': 1, 'click': 1, 'here': 1, 'prizes': 1})


## 1. Calculate Class Priors

Compute the prior probabilities for HAM and SPAM based on their frequency in the dataset.

In [3]:
# Calculate class priors
num_docs = len(documents)
num_ham = len(docs_by_class['HAM'])
num_spam = len(docs_by_class['SPAM'])

prior_ham = num_ham / num_docs
prior_spam = num_spam / num_docs

print(f"P(HAM) = {prior_ham:.2f}")
print(f"P(SPAM) = {prior_spam:.2f}")

P(HAM) = 0.55
P(SPAM) = 0.45


## 1. Calculate Token Likelihoods

Calculate the likelihood (probability) of each token in the vocabulary given each class, using Laplace smoothing.

In [4]:
# Calculate likelihoods with Laplace smoothing
vocab = sorted(vocab)
vocab_size = len(vocab)

likelihoods = {'HAM': {}, 'SPAM': {}}

for label in ['HAM', 'SPAM']:
    total_words = sum(word_counts[label].values())
    for word in vocab:
        # Laplace smoothing: add 1 to numerator, add |V| to denominator
        likelihoods[label][word] = (
            word_counts[label][word] + 1
        ) / (total_words + vocab_size)

# Display a few likelihoods for each class
import pandas as pd
pd.set_option('display.max_columns', None)

likelihoods_df = pd.DataFrame(likelihoods)
likelihoods_df

,HAM,SPAM
3,0.025316,0.014925
50,0.012658,0.029851
a,0.012658,0.029851
are,0.037975,0.014925
at,0.037975,0.014925
can,0.025316,0.014925
catch,0.025316,0.014925
click,0.012658,0.029851
dinner,0.025316,0.014925
for,0.025316,0.044776


## 1. Classify Test Sentences

Manually compute the posterior probabilities for the test sentences and assign the most probable class.

In [5]:
import numpy as np

def predict_naive_bayes(text):
    tokens = tokenize(text)
    # Only use tokens in vocab
    tokens = [t for t in tokens if t in vocab]
    log_prob = {}
    for label in ['HAM', 'SPAM']:
        # Start with log prior
        log_prob[label] = np.log(prior_ham if label == 'HAM' else prior_spam)
        for token in tokens:
            log_prob[label] += np.log(likelihoods[label][token])
    return log_prob

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

for sent in test_sentences:
    log_probs = predict_naive_bayes(sent)
    pred = max(log_probs, key=log_probs.get)
    print(f"Sentence: '{sent}'\n  HAM log-prob: {log_probs['HAM']:.2f}, SPAM log-prob: {log_probs['SPAM']:.2f} => Predicted: {pred}\n")

Sentence: 'Limited offer, click here!'
  HAM log-prob: -13.71, SPAM log-prob: -11.32 => Predicted: SPAM

Sentence: 'Meeting at 2 PM with the manager.'
  HAM log-prob: -13.81, SPAM log-prob: -17.61 => Predicted: HAM



## 2. Train Multinomial Naïve Bayes with Scikit-Learn

Use scikit-learn's CountVectorizer and MultinomialNB to train a Naïve Bayes classifier on the dataset.

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Prepare data for scikit-learn
X = [doc for doc, label in documents]
y = [label for doc, label in documents]

vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(X)

clf = MultinomialNB()
clf.fit(X_vec, y)

print("Vocabulary:", vectorizer.get_feature_names_out())

Vocabulary: ['50' 'are' 'at' 'can' 'catch' 'click' 'dinner' 'for' 'free' 'get' 'here'
 'hi' 'how' 'in' 'iphone' 'let' 'limited' 'lowest' 'meds' 'meeting' 'mom'
 'money' 'now' 'off' 'office' 'on' 'pm' 'price' 'prizes' 'report' 'send'
 'still' 'team' 'the' 'time' 'today' 'tomorrow' 'up' 'we' 'win' 'you'
 'your']


## 2. Classify Test Sentences Using Scikit-Learn

Use the trained scikit-learn model to predict the class of the test sentences and display the results.

In [7]:
# Predict the class of the test sentences using scikit-learn
X_test = test_sentences
X_test_vec = vectorizer.transform(X_test)
preds = clf.predict(X_test_vec)

for sent, pred in zip(X_test, preds):
    print(f"Sentence: '{sent}' => Predicted: {pred}")

Sentence: 'Limited offer, click here!' => Predicted: SPAM
Sentence: 'Meeting at 2 PM with the manager.' => Predicted: HAM
